# The Anatomy of AI · Lecture 12 · Neurons with Two Jobs — the notebook

Some neurons seem to fire on two unrelated things. In the lecture we tested three famous cases:
- **4e:55**, "cat faces and car fronts";
- **the car hidden in dog neurons**: 4c:447 feeds units that draw dogs;
- **CLIP's Spider-Man neuron**, which answers a photo, a drawing and a written word.

Here you rebuild the tests: both poles of a neuron over all 50,000 photos, drawn shapes, the wiring, a drawing that falls
into the "dog basin" and a route out of it, and CLIP read by a word on a paper label.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops.

**Runs on:** free Colab with a T4 GPU, in about fifteen minutes. It downloads InceptionV1 (27 MB), the lecture's own copy
of the ImageNet validation photos (two files of about 350 MB), and CLIP RN50x4 (about 400 MB) for the last part.

## 0. Setup
Installs (pinned, including OpenAI's CLIP at the commit the lecture used), the plot style, `check()`, the photos and InceptionV1.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3", "ftfy==6.3.1", "regex==2024.11.6", "tqdm==4.67.1", "packaging==24.2")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

pip("--no-deps", "https://github.com/openai/CLIP/archive/d05afc436d78f1c48dc0dbf8e5980a9d471f35f6.zip")   # OpenAI CLIP, the commit the lecture ran

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: two parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
import requests
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = f"val-{s}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(SHARD.format(s), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded photo file {s} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b, size=224):
    """JPEG bytes (256 px square) -> the centre 224 x 224, as a 3 x 224 x 224 tensor in [0, 1]."""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    if (w, h) != (256, 256):
        s = 256 / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        w, h = im.size
    l, t = (w - size) // 2, (h - size) // 2
    im = im.crop((l, t, l + size, t + size))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network this lecture studies, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

import contextlib
from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


class Taps:
    """Look inside the network. record=[module names] keeps their outputs in taps[name];
    edit={name: fn} replaces a module's output with fn(output) on the way through."""
    def __init__(self, record=(), edit=None):
        mods = dict(net.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))
        for name, fn in (edit or {}).items():
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, fn=fn: fn(o)))

    def __getitem__(self, name):
        return self.out[name]

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for h in self.h:
            h.remove()


# InceptionV1 has 1,008 outputs in its own order; PERM puts the 1,000 classes in the standard order (the lecture's
# perm.json, which also fixes two mislabelled outputs), and NAMES gives each class its name.
PERM = torch.tensor(json.loads("[449, 450, 442, 443, 444, 445, 446, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 494, 495, 496, 497, 498, 499, 500, 501, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 214, 215, 217, 216, 213, 218, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 226, 613, 614, 615, 616, 617, 618, 619, 620, 612, 422, 423, 424, 425, 427, 426, 428, 429, 430, 438, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 6, 22, 193, 14, 173, 99, 87, 69, 116, 198, 43, 89, 200, 98, 161, 132, 32, 180, 42, 18, 207, 181, 105, 77, 189, 20, 204, 63, 31, 66, 21, 25, 45, 170, 119, 210, 107, 126, 88, 145, 59, 160, 152, 72, 33, 91, 4, 27, 113, 123, 36, 156, 109, 158, 131, 26, 71, 56, 146, 144, 125, 176, 139, 134, 90, 2, 192, 154, 150, 94, 5, 19, 191, 196, 117, 141, 68, 93, 51, 208, 184, 97, 29, 171, 124, 128, 47, 64, 211, 112, 187, 114, 86, 115, 79, 130, 140, 110, 82, 17, 177, 149, 15, 3, 41, 127, 84, 143, 133, 60, 172, 179, 118, 168, 148, 70, 197, 50, 106, 49, 151, 46, 205, 102, 28, 58, 155, 136, 202, 34, 62, 1, 159, 67, 174, 55, 10, 95, 8, 11, 201, 85, 153, 30, 190, 76, 206, 61, 163, 103, 209, 74, 35, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 224, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 225, 657, 658, 188, 129, 164, 157, 13, 53, 183, 195, 101, 39, 80, 147, 78, 120, 167, 108, 162, 165, 81, 52, 9, 65, 57, 12, 121, 186, 48, 23, 182, 40, 212, 44, 16, 178, 38, 83, 104, 96, 185, 122, 73, 135, 137, 138, 203, 54, 100, 175, 142, 166, 37, 111, 92, 199, 75, 194, 24, 7, 169, 447, 451, 448, 452, 453, 456, 457, 454, 455, 547, 853, 896, 223, 345, 246, 230, 232, 677, 265, 264, 522, 688, 845, 752, 540, 847, 704, 767, 233, 907, 967, 341, 718, 916, 307, 705, 683, 518, 905, 258, 807, 908, 296, 353, 785, 909, 884, 266, 733, 991, 849, 777, 811, 933, 941, 254, 985, 835, 533, 839, 689, 252, 940, 805, 300, 706, 779, 538, 817, 716, 872, 719, 949, 851, 820, 580, 833, 887, 707, 267, 673, 591, 539, 239, 377, 836, 576, 568, 891, 749, 563, 548, 890, 929, 701, 241, 987, 342, 914, 827, 721, 902, 382, 762, 303, 335, 301, 801, 690, 695, 370, 713, 797, 979, 761, 996, 674, 698, 583, 543, 708, 243, 268, 376, 347, 910, 881, 297, 545, 778, 898, 298, 670, 756, 856, 865, 720, 313, 550, 959, 966, 523, 529, 315, 821, 667, 579, 715, 253, 897, 972, 834, 336, 799, 1000, 662, 512, 346, 262, 316, 879, 660, 808, 796, 305, 235, 279, 919, 995, 356, 309, 784, 261, 712, 934, 299, 256, 348, 671, 757, 280, 971, 567, 955, 275, 792, 276, 236, 337, 911, 227, 684, 725, 703, 517, 581, 895, 249, 375, 840, 505, 551, 750, 573, 350, 344, 554, 369, 787, 696, 730, 562, 802, 924, 293, 525, 980, 659, 592, 748, 269, 961, 963, 294, 560, 770, 773, 582, 918, 892, 814, 228, 374, 988, 371, 687, 238, 546, 270, 244, 867, 973, 894, 508, 536, 697, 532, 818, 917, 782, 977, 763, 338, 339, 781, 984, 598, 922, 946, 302, 717, 509, 661, 866, 875, 920, 880, 271, 251, 871, 829, 290, 272, 764, 686, 869, 277, 824, 854, 691, 852, 260, 255, 728, 511, 599, 284, 588, 585, 595, 755, 915, 552, 699, 354, 351, 520, 513, 333, 870, 937, 295, 506, 921, 826, 564, 584, 504, 759, 685, 352, 877, 942, 994, 306, 527, 257, 679, 843, 732, 842, 850, 883, 783, 789, 575, 926, 722, 281, 596, 931, 901, 888, 841, 565, 245, 983, 372, 693, 964, 731, 381, 378, 857, 923, 285, 855, 314, 788, 838, 566, 373, 769, 556, 702, 542, 534, 572, 846, 939, 862, 976, 273, 860, 571, 863, 537, 927, 286, 570, 965, 668, 578, 694, 219, 541, 310, 663, 997, 876, 519, 760, 753, 586, 952, 751, 938, 355, 809, 521, 962, 221, 729, 510, 587, 379, 589, 559, 800, 709, 832, 950, 259, 380, 868, 747, 590, 776, 943, 549, 726, 557, 507, 288, 278, 960, 222, 986, 577, 925, 822, 858, 515, 234, 676, 237, 600, 775, 274, 593, 804, 263, 680, 340, 530, 998, 724, 528, 516, 514, 287, 957, 311, 692, 247, 794, 526, 574, 535, 810, 681, 828, 837, 945, 569, 561, 531, 304, 250, 978, 675, 786, 944, 970, 989, 903, 758, 555, 308, 780, 664, 710, 312, 594, 700, 282, 711, 289, 283, 766, 825, 291, 242, 597, 678, 882, 349, 765, 727, 544, 220, 292, 334, 666, 874, 990, 969, 558, 790, 682, 343, 936, 665, 524, 928, 317, 231, 906, 669, 958, 819, 795, 772, 502, 899, 912, 904, 935, 831, 503, 672, 951, 815, 723, 248, 240, 714, 553, 930, 791, 932, 861, 774, 803, 754, 813, 844, 771, 793, 974, 968, 873, 768, 975, 993, 885, 734, 736, 737, 738, 739, 740, 741, 742, 743, 744, 735, 745, 746, 318, 229, 319, 320, 321, 322, 323, 324, 325, 326, 816, 999, 953, 864, 806, 948, 830, 900, 798, 947, 859, 823, 361, 992, 359, 365, 368, 366, 363, 364, 367, 360, 362, 954, 848, 982, 330, 357, 358, 331, 327, 328, 332, 886, 913, 956, 893, 878, 812, 981, 329, 889]"), device=DEV)
from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]

In [ ]:
ALL_LABELS = np.concatenate([_shard(s).read(columns=["label"]).column("label").to_numpy() for s in (0, 1)]).astype(int)
mods = dict(net.named_modules())
BRANCHES = {"mixed4c": (128, 256, 64, 64), "mixed4d": (112, 288, 64, 64), "mixed4e": (256, 320, 128, 128),
            "mixed5a": (256, 320, 128, 128)}
BRANCH_NAMES = ("1x1", "3x3", "5x5", "pool_reduce")

def branch_of(layer, ch):
    start = 0
    for name, n in zip(BRANCH_NAMES, BRANCHES[layer]):
        if ch < start + n:
            return name, ch - start
        start += n

def pre_relu(layer, ch):
    """(module name, index) of the conv whose output is this channel's value before its gate."""
    b, i = branch_of(layer, ch)
    return f"{layer}_{b}_pre_relu_conv", i

# The lecture's class groups (standard class numbers)
DOGS = list(range(151, 269)); CATS = list(range(281, 286)); BIGCATS = list(range(286, 294))
FOXES = list(range(277, 281)); WOLVES = list(range(269, 273))
CARS = [407, 436, 468, 511, 609, 627, 656, 661, 717, 751, 817, 734, 675, 864, 654, 581, 479]
VEHICLES = CARS + [555, 569, 573, 603, 705, 779, 803, 820, 829, 847, 867, 874, 466, 565, 671, 670, 880]
frac = lambda ids, pool: float(np.isin(ALL_LABELS[ids], pool).mean())

## 1. The famous one, and its other pole (Plates I–IV)
4e:55 sits in mixed4e's 1 × 1 branch. A neuron's value *before its gate* (the ReLU) can be negative too: the gate hides
that half, but the neuron computes it. One pass over all 50,000 photos keeps, for each photo:
- 4e:55's strongest and weakest value before the gate, and its average over the whole 14 × 14 map;
- the units we need later: the car detector 4c:447, its readers in mixed4d, and a few mixed4e units.

In [ ]:
D4 = [417, 167, 295, 241, 277, 352, 455]
E4 = [651, 629, 820, 729, 46, 688, 670, 53]              # section 5's joint readers
p55 = pre_relu("mixed4e", 55)
pmax = np.zeros(50000, np.float32); pmin = np.zeros(50000, np.float32); pmean = np.zeros(50000, np.float32)
mx447 = np.zeros(50000, np.float32); mx4d = np.zeros((50000, len(D4)), np.float32); mx4e = np.zeros((50000, len(E4)), np.float32)
t = time.time()
with torch.no_grad(), Taps([p55[0], "mixed4c", "mixed4d", "mixed4e"]) as T:
    for s in (0, 1):
        f = _shard(s)
        for g in range(f.num_row_groups):
            ims = f.read_row_group(g, columns=["image"]).column("image").to_pylist()
            net(prep(torch.stack([decode(im["bytes"]) for im in ims]).to(DEV)))
            n0, n1 = 25000 * s + 100 * g, 25000 * s + 100 * g + len(ims)
            a = T[p55[0]][:, p55[1]].flatten(1)
            pmax[n0:n1], pmin[n0:n1], pmean[n0:n1] = a.amax(1).cpu().numpy(), a.amin(1).cpu().numpy(), a.mean(1).cpu().numpy()
            mx447[n0:n1] = T["mixed4c"][:, 447].amax((1, 2)).cpu().numpy()
            mx4d[n0:n1] = T["mixed4d"][:, D4].amax((2, 3)).cpu().numpy()
            mx4e[n0:n1] = T["mixed4e"][:, E4].amax((2, 3)).cpu().numpy()
print(f"50,000 photos in {time.time() - t:.0f} s")

In [ ]:
top = np.argsort(-pmax)[:200]; bot = np.argsort(pmin)[:200]; botm = np.argsort(pmean)[:200]
print("strongest 12:", [NAMES[ALL_LABELS[n]] for n in top[:12]])
print("most negative 12 (by its weakest place):", [NAMES[ALL_LABELS[n]] for n in bot[:12]])
print("most negative 12 (by its average):", [NAMES[ALL_LABELS[n]] for n in botm[:12]])
L = {"cat": 0.295, "fox_wolf": 0.175, "dog": 0.2, "car": 0.065}
check("of its strongest 200: cats (house and big)", frac(top, CATS + BIGCATS), L["cat"], tol=0.011)
check("foxes and wolves", frac(top, FOXES + WOLVES), L["fox_wolf"], tol=0.011)
check("dogs", frac(top, DOGS), L["dog"], tol=0.011)
check("cars", frac(top, CARS), L["car"], tol=0.011)
check("of its most negative 200 (weakest place): dogs", frac(bot, DOGS), 0.595, tol=0.011)
check("... and cats", frac(bot, CATS + BIGCATS), 0.0, tol=0.006)
check("most negative by its average: the first three", [NAMES[ALL_LABELS[n]] for n in botm[:3]], ["slide rule", "bucket", "barometer"])

In [ ]:
fig, axs = plt.subplots(2, 6, figsize=(11, 4.3))
rw = rows([int(n) for n in list(top[:6]) + list(bot[:6])])
for a, n, tag in zip(axs.flat, list(top[:6]) + list(bot[:6]), ["strongest"] * 6 + ["most negative"] * 6):
    a.imshow(decode(rw[int(n)][0]).permute(1, 2, 0).numpy()); a.axis("off")
    a.set_title(f"{NAMES[ALL_LABELS[n]][:15]}", fontsize=8)
axs[0, 0].set_ylabel("strongest"); fig.suptitle("4e:55: top row its strongest photos, bottom row its most negative", fontsize=11)
plt.show()

# Pole to pole: every photo's average value before the gate
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.hist(pmean, bins=200, color=PAPER, edgecolor=INK, lw=0.4)
ax.set_yscale("log"); ax.set_xlabel("4e:55's average value before the gate, per photo"); ax.set_ylabel("photos")
plt.show()

**Class statistics (Plate IV).** Rank all 1,000 classes by 4e:55's average peak over their 50 photos.

In [ ]:
cm = np.array([pmax[ALL_LABELS == k].mean() for k in range(1000)])
o = np.argsort(-cm)
rank = {k: int(np.where(o == k)[0][0]) + 1 for k in range(1000)}
print("top ten classes:", [NAMES[k] for k in o[:10]])
groups = {"cats": CATS, "foxes": FOXES, "cars": CARS, "wolves": WOLVES, "big cats": BIGCATS, "dogs": DOGS}
LECT = {"cats": 6.0, "big cats": 77.0, "foxes": 9.5, "wolves": 46.5, "dogs": 217.5, "cars": 27.0}
for g, pool in groups.items():
    print(f"  {g:9s} median rank {np.median([rank[k] for k in pool]):6.1f}")
check("its no. 1 class", NAMES[o[0]], ["papillon", "red fox", "Persian cat", "tabby", "Pomeranian", "tiger cat", "beach wagon", "Japanese spaniel", "kit fox", "Arctic fox"][0])
check("its no. 2 class", NAMES[o[1]], ["papillon", "red fox", "Persian cat", "tabby", "Pomeranian", "tiger cat", "beach wagon", "Japanese spaniel", "kit fox", "Arctic fox"][1])
check("the lecture's top ten, how many are in ours", len(set(NAMES[k] for k in o[:10]) & set(["papillon", "red fox", "Persian cat", "tabby", "Pomeranian", "tiger cat", "beach wagon", "Japanese spaniel", "kit fox", "Arctic fox"])), 10, tol=1)
for g, pool in groups.items():
    check(f"{g}: median rank of 1,000", float(np.median([rank[k] for k in pool])), LECT[g], tol=1.5)

A small pointy-faced dog and the red fox come first; foxes rank with cats; round things sit near the bottom.

## 2. Pointed or round? (Plate V)
If 4e:55 means "pointed vs round", drawn shapes should show it. We draw 456 shapes at the place its middle unit looks:
- V chevrons pointing 12 ways, at 3 openings;
- U shapes and O rings and discs;
- each at 3 sizes, dark on light and light on dark, as outlines and filled.

Then we read its value there, before the gate. First, find where the middle unit (row 7, column 7) looks, by gradient.

In [ ]:
g = torch.full((1, 3, 224, 224), 0.5, device=DEV, requires_grad=True)
with Taps(["mixed4e_1x1_pre_relu_conv"]) as T:
    net(prep(g)); T["mixed4e_1x1_pre_relu_conv"][0, 55, 7, 7].backward()
G = g.grad[0].abs().sum(0).cpu().numpy(); yy, xx = np.mgrid[0:224, 0:224]
ox, oy = float((xx * G).sum() / G.sum()), float((yy * G).sum() / G.sum())
print(f"our centre: ({ox:.1f}, {oy:.1f}); the lecture's: ({[131.8, 128.8][0]}, {[131.8, 128.8][1]})")
# On a plain grey picture every value inside a max-pool is equal, so where the gradient goes depends on how the GPU
# breaks those ties: the centre moves by a few pixels between GPUs. We draw at the lecture's centre so the numbers
# below compare with its own.
check("our centre is within 6 px of the lecture's", math.hypot(ox - [131.8, 128.8][0], oy - [131.8, 128.8][1]), 0, tol=6)
cx, cy = [131.8, 128.8]

from PIL import ImageDraw
def draw(kind, size, point_deg=270.0, opening=60.0, fill=False, ink=0, bg=255, width=4, ss=4):
    S = 224 * ss; im = Image.new("L", (S, S), bg); d = ImageDraw.Draw(im)
    c = np.array([cx, cy]) * ss; r = size * ss / 2
    if kind == "O":
        box = [c[0] - r, c[1] - r, c[0] + r, c[1] + r]
        d.ellipse(box, fill=ink) if fill else d.ellipse(box, outline=ink, width=width * ss)
    elif kind == "V":
        a = math.radians(point_deg); tip = c + np.array([math.cos(a), -math.sin(a)]) * r * 0.8; h = math.radians(opening / 2)
        arms = [tip + np.array([math.cos(a + math.pi + s * h), -math.sin(a + math.pi + s * h)]) * r * 1.6 / max(math.cos(h), 0.3) for s in (-1, 1)]
        if fill:
            d.polygon([tuple(tip), tuple(arms[0]), tuple(arms[1])], fill=ink)
        else:
            d.line([tuple(arms[0]), tuple(tip), tuple(arms[1])], fill=ink, width=width * ss, joint="curve")
    else:   # U: a semicircle where the V has its point
        a = math.radians(point_deg); back = c - np.array([math.cos(a), -math.sin(a)]) * r * 0.2
        st_ = -math.degrees(a) - 90
        d.arc([back[0] - r, back[1] - r, back[0] + r, back[1] + r], st_, st_ + 180, fill=ink, width=width * ss)
    im = im.resize((224, 224), Image.LANCZOS)
    return torch.from_numpy(np.asarray(im, np.float32) / 255.0)[None].repeat(3, 1, 1)

stim = []
for size in (48, 80, 120):
    for pol in ("dark", "light"):
        ink, bg = (0, 255) if pol == "dark" else (255, 0)
        for fill in (False, True):
            stim.append((("O", size, pol, fill, 0, 0), draw("O", size, fill=fill, ink=ink, bg=bg)))
            stim.append((("U", size, pol, fill, 270, 0), draw("U", size, 270, fill=False, ink=ink, bg=bg)))
            for opening in (30, 60, 90):
                for pd in range(0, 360, 30):
                    stim.append((("V", size, pol, fill, pd, opening), draw("V", size, pd, opening, fill=fill, ink=ink, bg=bg)))
vals = []
with torch.no_grad(), Taps(["mixed4e_1x1_pre_relu_conv"]) as T:
    for i in range(0, len(stim), 64):
        net(prep(torch.stack([s[1] for s in stim[i:i + 64]]).to(DEV)))
        vals += T["mixed4e_1x1_pre_relu_conv"][:, 55, 7, 7].cpu().tolist()
kinds = np.array([s[0][0] for s in stim]); vals = np.array(vals)
check("shapes drawn", len(stim), 456)
for k in "VUO":
    print(f"  {k}: median {np.median(vals[kinds == k]):+.1f}  (min {vals[kinds == k].min():+.1f}, max {vals[kinds == k].max():+.1f})")
    check(f"{k} shapes: median value", float(np.median(vals[kinds == k])), {"V": 12.8, "U": 7.18, "O": -8.44}[k], tol=0.3)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1, 2.2]})
ex = [s for s in stim if s[0] in {("V", 80, "dark", False, 270, 60), ("U", 80, "dark", False, 270, 0), ("O", 80, "dark", False, 0, 0)}]
axs[0].imshow(torch.cat([e[1][0] for e in ex], 1).numpy(), cmap="gray", vmin=0, vmax=1); axs[0].axis("off")
axs[0].set_title("V, U, O (three of 456)", fontsize=9)
for j, k in enumerate("VUO"):
    v = vals[kinds == k]
    axs[1].plot(v, np.full(len(v), j) + np.random.default_rng(j).uniform(-0.25, 0.25, len(v)), ".", ms=2.5, color=INK2)
    axs[1].plot([np.median(v)] * 2, [j - 0.35, j + 0.35], color=INK, lw=2.5)
axs[1].set_yticks([0, 1, 2], ["V", "U", "O"]); axs[1].axvline(0, color=INK, lw=0.6, ls="--")
axs[1].set_xlabel("4e:55 at its middle place, before the gate")
plt.show()

V's above U's above O's, in the direction "pointed vs round" predicts, but weakly: photos push it ten times harder.

## 3. Its wiring (Plate VI)
4e:55 is a 1 × 1 unit, so its weights from the 528 maps of mixed4d are 528 single numbers. Who feeds it most? Then the
other way: which mixed5a units read it most, through their *expanded weights* (the bottleneck times the filter, as in
Lecture 11)?

In [ ]:
w55 = mods["mixed4e_1x1_pre_relu_conv"].weight[55, :, 0, 0].cpu().numpy()
ow = np.argsort(-w55)
print("strongest inputs:", [(int(u), round(float(w55[u]), 3)) for u in ow[:8]])
check("its strongest input", int(ow[0]), [493, 0.3113][0])
check("that weight", float(w55[ow[0]]), [493, 0.3113][1], tol=1e-4)
check("the next strongest weight", float(w55[ow[1]]), 0.1279, tol=1e-4)
for u, r in {"493": 1, "477": 8, "201": 15}.items():
    check(f"4d:{u}'s rank by weight", int(np.where(ow == int(u))[0][0]) + 1, r)

# by weight x activation: each input's push at 4e:55's peak, averaged over its 200 strongest photos
rw = rows([int(n) for n in top])
contr = []
with torch.no_grad(), Taps(["mixed4d", "mixed4e_1x1_pre_relu_conv"]) as T:
    for s in range(0, 200, 100):
        net(prep(torch.stack([decode(rw[int(n)][0]) for n in top[s:s + 100]]).to(DEV)))
        at = T["mixed4e_1x1_pre_relu_conv"][:, 55].flatten(1).argmax(1)
        d4 = T["mixed4d"].flatten(2)[torch.arange(len(at)), :, at]
        contr.append((d4 * torch.tensor(w55, device=DEV)[None]).cpu())
contr = torch.cat(contr).mean(0).numpy(); oc = np.argsort(-contr)
for u, r in {"493": 1, "477": 2, "201": 36}.items():
    check(f"4d:{u}'s rank by weight x activation", int(np.where(oc == int(u))[0][0]) + 1, r, tol=1)

In [ ]:
def expanded_all(layer):
    """[units of layer, maps below, 5, 5]: every unit's expanded weights (smaller kernels padded to 5 x 5)."""
    out = []
    for b in BRANCH_NAMES:
        W = mods[f"{layer}_{b}_pre_relu_conv"].weight.cpu()
        if b in ("3x3", "5x5"):
            W = torch.einsum("obhw,bp->ophw", W, mods[f"{layer}_{b}_bottleneck_pre_relu_conv"].weight[:, :, 0, 0].cpu())
        pad = (5 - W.shape[-1]) // 2
        out.append(F.pad(W, (pad, pad, pad, pad)))
    return torch.cat(out).numpy()

E5a = expanded_all("mixed5a")                       # 832 x 832 x 5 x 5
from55 = E5a[:, 55].sum((1, 2)); o5 = np.argsort(-from55)
print("mixed5a units that read 4e:55 most:", [(int(u), round(float(from55[u]), 3)) for u in o5[:5]])
check("4e:55's strongest reader", int(o5[0]), 352)
for u, r in ((352, 1), (94, 2), (317, 30)):
    check(f"4e:55's rank among 5a:{u}'s 832 inputs", int(np.where(np.argsort(-E5a[u].sum((1, 2))) == 55)[0][0]) + 1, r)

Its strongest input, 4d:493, is a fox-snout unit. It feeds a coyote-face reader, and it is the second input of a reader whose photos include candles and matchsticks: pointed things.

## 4. The car's readers (Plates VII–VIII)
Lecture 11's car detector 4c:447 feeds mixed4d. Which of mixed4d's 528 units read it most strongly? Zoom In and the essay
name four that it feeds: 4d:417, 167, 295 and 241. Then: what do those units fire on, among their strongest photos?

In [ ]:
E4d = expanded_all("mixed4d")                       # 528 x 512 x 5 x 5
from447 = E4d[:, 447].sum((1, 2)); od = np.argsort(-from447)
print("4c:447's strongest readers:", [(int(u), round(float(from447[u]), 3)) for u in od[:7]])
for (u, wv), (u2, w2) in zip([(int(u), float(from447[u])) for u in od[:7]], [[417, 0.7671], [352, 0.5165], [455, 0.4617], [295, 0.4152], [167, 0.4111], [277, 0.4085], [241, 0.3985]]):
    check(f"reader: 4d:{u2}", u, u2); check(f"  its weight from the car", wv, w2, tol=1e-3)

STATS = {"4c:447": {"top40": [1.0, 0.0], "top200": [0.96, 0.0], "top1000": [0.777, 0.005]}, "4d:417": {"top40": [1.0, 0.0], "top200": [0.845, 0.085], "top1000": [0.387, 0.387]}, "4d:167": {"top40": [0.975, 0.025], "top200": [0.87, 0.03], "top1000": [0.492, 0.182]}, "4d:295": {"top40": [0.675, 0.3], "top200": [0.635, 0.36], "top1000": [0.4, 0.545]}, "4d:241": {"top40": [0.475, 0.525], "top200": [0.495, 0.49], "top1000": [0.32, 0.614]}, "4d:277": {"top40": [0.525, 0.0], "top200": [0.575, 0.015], "top1000": [0.389, 0.047]}}
series = {"4c:447": mx447, **{f"4d:{u}": mx4d[:, j] for j, u in enumerate(D4)}}
for name in ("4d:417", "4d:167", "4d:295", "4d:241", "4c:447"):
    v = series[name]
    for k in (40, 200, 1000):
        idx = np.argsort(-v)[:k]
        veh, dog = frac(idx, VEHICLES), frac(idx, DOGS)
        if k == 40 or name in ("4d:295", "4d:241"):
            check(f"{name}, top {k}: vehicles", veh, STATS[name][f"top{k}"][0], tol=0.026 if k == 40 else 0.011)
            check(f"{name}, top {k}: dogs", dog, STATS[name][f"top{k}"][1], tol=0.026 if k == 40 else 0.011)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
labs = ["4c:447", "4d:417", "4d:167", "4d:295", "4d:241"]
for j, name in enumerate(labs):
    idx = np.argsort(-series[name])[:40]
    veh, dog = frac(idx, VEHICLES), frac(idx, DOGS)
    ax.barh(j, veh, color=INK, edgecolor=INK); ax.barh(j, dog, left=veh, color=PAPER, edgecolor=INK, hatch="////")
ax.set_yticks(range(len(labs)), labs); ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.set_xlabel("share of its 40 strongest photos of 50,000: vehicles (black), dogs (hatched)")
plt.show()

## 5. Draw them: into the dog basin, and out (Plates IX–XI)
*Feature visualization*: start from noise and change the picture, step by step, so that a neuron fires more. Drawn
freely, 4d:241 draws a dog. The lecture's fix was to route the drawing **through the car**: for the first 64 of 512
steps, every mixed4c map except the car detector 4c:447 is switched off on the way in, so the only way to make 4d:241
fire is through the car. After that the drawing is free.

The picture is built from random frequencies, low ones louder, in decorrelated colour, as Lucid did in 2017. Each step
it is jittered, scaled and turned a little, so it can't cheat with pixel noise. (This from-scratch version follows Part
VI-b's notebook. The lecture used lucent's own, so the numbers will differ, but the story should not.)

In [ ]:
COLOR = np.array([[0.26, 0.09, 0.02], [0.27, 0.00, -0.05], [0.27, -0.09, 0.03]], dtype=np.float32)
COLOR = COLOR / np.max(np.linalg.norm(COLOR, axis=0))

def fft_image(B, h=224, w=224, sd=0.01, seed=0):
    g_ = torch.Generator().manual_seed(seed)
    fy, fx = np.fft.fftfreq(h)[:, None], np.fft.fftfreq(w)[: w // 2 + 1]
    freqs = np.sqrt(fx * fx + fy * fy)
    spec = (torch.randn((B, 3) + freqs.shape + (2,), generator=g_) * sd).to(DEV).requires_grad_(True)
    scale = torch.tensor(1.0 / np.maximum(freqs, 1.0 / max(w, h))).float()[None, None, ..., None].to(DEV)
    Cm = torch.tensor(COLOR, device=DEV)
    def image():
        x = torch.fft.irfftn(torch.view_as_complex(scale * spec), s=(h, w), norm="ortho") / 4.0
        return torch.sigmoid(torch.einsum("bchw,dc->bdhw", x, Cm))
    return spec, image

def jiggle(x):
    B_ = x.shape[0]
    x = F.pad(x, (12, 12, 12, 12), value=0.5)
    def jitter(x, d):
        dx, dy = np.random.randint(0, d + 1, 2)
        return x[:, :, dy:x.shape[2] - (d - dy), dx:x.shape[3] - (d - dx)]
    x = jitter(x, 8)
    x = F.interpolate(x, scale_factor=float(np.random.choice([1 + (i - 5) / 50.0 for i in range(11)])), mode="bilinear", align_corners=False)
    a = math.radians(float(np.random.choice(list(range(-10, 11)) + 5 * [0])))
    th = torch.tensor([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0]], device=x.device).float()
    x = F.grid_sample(x, F.affine_grid(th[None].expand(B_, 2, 3), list(x.shape), align_corners=False), padding_mode="border", align_corners=False)
    x = jitter(x, 4)
    return F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False) if min(x.shape[-2:]) < 224 else x

p241 = pre_relu("mixed4d", 241)
only447 = torch.zeros(1, 512, 1, 1, device=DEV); only447[0, 447] = 1
route = {"on": False}
def through_car(o):
    return o * only447 if route["on"] else o

def drawing(through, steps=512, seed=0):
    np.random.seed(seed); torch.manual_seed(seed)
    spec, image = fft_image(1, seed=seed)
    opt = torch.optim.Adam([spec], lr=0.05)
    trace = []
    with Taps([p241[0]], edit={"mixed4c": through_car}) as T:
        for step in range(steps):
            route["on"] = through and step < 64
            opt.zero_grad()
            net(prep(jiggle(image())))
            (-T[p241[0]][0, p241[1], 7, 7]).backward(); opt.step()
            if step % 32 == 31:
                route["on"] = False
                with torch.no_grad(), Taps(["mixed4c", "mixed4d"]) as T2:
                    net(prep(image()))
                    trace.append((step + 1, float(T2["mixed4d"][0, 241].max()), float(T2["mixed4c"][0, 447].max())))
    route["on"] = False
    with torch.no_grad():
        x = image()
        says = NAMES[int(net(prep(x))[0, PERM].argmax())]
    return x[0].permute(1, 2, 0).cpu().numpy(), trace, says

t = time.time()
free_img, free_trace, free_says = drawing(False)
car_img, car_trace, car_says = drawing(True)
print(f"two drawings in {time.time() - t:.0f} s")
print(f"free:            4d:241 {free_trace[-1][1]:7.0f}, car detector {free_trace[-1][2]:7.0f}; InceptionV1 calls it a '{free_says}'")
print(f"through the car: 4d:241 {car_trace[-1][1]:7.0f}, car detector {car_trace[-1][2]:7.0f}; InceptionV1 calls it a '{car_says}'")
VEH_NAMES = {NAMES[k] for k in VEHICLES} | {"grille", "car wheel"}
check("the free drawing is not a vehicle", free_says in VEH_NAMES, False)
check("the drawing through the car is a vehicle", car_says in VEH_NAMES, True)
check("4d:241 fires about as much on both (ratio within 0.5-2)", bool(0.5 < car_trace[-1][1] / free_trace[-1][1] < 2), True)
check("the car detector fires at least 5x more on the drawing through the car", bool(car_trace[-1][2] > 5 * max(free_trace[-1][2], 1.0)), True)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 1, 1.6]})
axs[0].imshow(free_img); axs[0].set_title(f"4d:241, free: '{free_says}'", fontsize=9)
axs[1].imshow(car_img); axs[1].set_title(f"through the car: '{car_says}'", fontsize=9)
for a in axs[:2]: a.axis("off")
for tr, ls, lab in ((free_trace, "--", "free"), (car_trace, "-", "through the car")):
    axs[2].plot([s for s, _, _ in tr], [c for _, _, c in tr], ls, color=INK, label=f"car detector, {lab}")
axs[2].axvline(64, color=INK2, lw=0.8, ls=":"); axs[2].text(70, axs[2].get_ylim()[1] * 0.45, "the route\nends here", fontsize=8, color=INK2)
axs[2].set_xlabel("step"); axs[2].set_ylabel("4c:447 on the drawing"); axs[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

The lecture's numbers, with lucent's drawing code: free, 4d:241 2,116 and the car detector 309; through the car, 2,144
and 6,076. The unit fires about equally on a dog and on a car, while the car detector differs twenty-fold. Free
drawing falls into the easier "dog basin"; the route through the car finds the car, and the drawing stays a car.

## 6. Is anything reading the hidden car? (Plate XII)
If a car were stored across 417, 167, 295 and 241, some unit in the next layer should read all four together and fire
on cars. Find the mixed4e units with a positive expanded weight from all four, and look at their strongest photos.

In [ ]:
E4e = expanded_all("mixed4e")                       # 832 x 528 x 5 x 5
W4 = E4e[:, [417, 167, 295, 241]].sum((2, 3)); jointly = W4.min(1); oj = np.argsort(-jointly)
check("mixed4e units reading all four positively", int((jointly > 0).sum()), 21)
for (u, veh) in [[651, 0.0], [629, 0.11], [820, 0.04], [729, 0.02], [46, 0.88], [688, 0.0], [670, 0.035], [53, 0.11]]:
    j = E4.index(u)
    v = frac(np.argsort(-mx4e[:, j])[:200], VEHICLES)
    print(f"  4e:{u}: weights from the four " + ", ".join(f"{w_:+.3f}" for w_ in W4[u]) + f"; vehicles among its top 200: {v:.2f}")
    check(f"4e:{u}, vehicles among its top 200", v, veh, tol=0.011)
check("the strongest joint readers, in order", [int(u) for u in oj[:8]], [u for u, _ in [[651, 0.0], [629, 0.11], [820, 0.04], [729, 0.02], [46, 0.88], [688, 0.0], [670, 0.035], [53, 0.11]]])

Only one joint reader, 4e:46, is a vehicle unit, and its weight from the four comes almost all from 4d:417, itself a car unit. Nothing reads the combination back out as a car.

## 7. A network that read the internet (Plates XIII–XVI)
**CLIP** learned from 400 million photos and their captions. To classify a photo, compare it with a sentence for each
class ("a photo of the small Granny Smith." …, seven ways of saying it, averaged) and pick the closest. Unit 550 of its
last convolution layer is the one the paper called a Spider-Man neuron. We measure its average over the map.

The lecture averaged three fonts; this notebook uses the first, DejaVu Sans Bold, which ships with matplotlib. The
lecture's numbers for that font alone come from running its own code once (`expect_l12.py`).

In [ ]:
import clip, matplotlib
from PIL import ImageFont
FONT = os.path.join(matplotlib.get_data_path(), "fonts", "ttf", "DejaVuSans-Bold.ttf")
with contextlib.redirect_stderr(io.StringIO()):
    cmodel, cpre = clip.load("RN50x4", device=DEV, download_root="clip_weights")
cmodel.eval()
print("CLIP RN50x4: ready")
TEMPLATES = ["itap of a {}.", "a bad photo of the {}.", "a origami {}.", "a photo of the large {}.",
             "a {} in a video game.", "art of the {}.", "a photo of the small {}."]
with torch.no_grad():
    Z = []
    for n in NAMES:
        e = cmodel.encode_text(clip.tokenize([tp.format(n) for tp in TEMPLATES]).to(DEV)).float()
        e = e / e.norm(dim=-1, keepdim=True); e = e.mean(0); Z.append(e / e.norm())
    Z = torch.stack(Z)
feats = {}
cmodel.visual.layer4.register_forward_hook(lambda m_, i, o: feats.__setitem__("l4", o))

def classify(ims):
    x = torch.stack([cpre(im) for im in ims]).to(DEV)
    with torch.no_grad():
        f = cmodel.encode_image(x.type(cmodel.dtype)).float(); f = f / f.norm(dim=-1, keepdim=True)
        p = (100.0 * f @ Z.T).softmax(-1)
    return p.cpu().numpy(), feats["l4"].float().mean((2, 3)).cpu().numpy()

def photo_pil(jpg):
    """The middle 224 x 224 of a photo, as a PIL image (the lecture's crop)."""
    im = Image.open(io.BytesIO(jpg)).convert("RGB"); l = (im.size[0] - 224) // 2
    return im.crop((l, l, l + 224, l + 224))

def word(t, size=300):
    im = Image.new("RGB", (size, size), (255, 255, 255)); d = ImageDraw.Draw(im); s = 64
    f = ImageFont.truetype(FONT, s)
    while d.textlength(t, font=f) > size * 0.9 and s > 8:
        s -= 2; f = ImageFont.truetype(FONT, s)
    d.text(((size - d.textlength(t, font=f)) / 2, size / 2 - s * 0.6), t, fill=(0, 0, 0), font=f)
    return im

WORDS = {"Spider-Man": 1.27, "spider-man": 1.2998, "SPIDERMAN": 1.2306, "spider": 0.9013, "Peter Parker": 0.0088, "spider web": 1.2428, "Batman": 0.0163, "Superman": 0.1477, "Iron Man": 0.0419, "banana": 0.013, "dog": 0.0262, "lamp": 0.0029, "Granny Smith": 0.0514, "teapot": 0.0172, "Hulk": 0.051, "web browser": 0.3129}
_, a = classify([word(t) for t in WORDS])
for (t, want), v in zip(WORDS.items(), a[:, 550]):
    print(f"  '{t}': unit 550 = {v:.3f}")
    check(f"unit 550 on the word '{t}'", float(v), want, tol=max(0.02, 0.03 * want))

In [ ]:
# Photos: the spider classes and spider webs, 25 photos each, against 100 other classes chosen with numpy seed 0
spider_classes = sorted(set([k for k, n in enumerate(NAMES) if "spider" in n.lower()] + [NAMES.index("spider web")]))
rng = np.random.default_rng(0)
others = sorted(rng.choice([k for k in range(1000) if k not in spider_classes], 100, replace=False).tolist())
first25 = {k: np.where(ALL_LABELS == k)[0][:25].tolist() for k in spider_classes + others}
prw = rows([n for v in first25.values() for n in v])
cls_mean = {}
for k, ids in first25.items():
    _, a = classify([photo_pil(prw[n][0]) for n in ids]); cls_mean[NAMES[k]] = float(a[:, 550].mean())
srt = sorted(cls_mean.items(), key=lambda kv: -kv[1])
print("unit 550's top classes (25 photos each):", [(n, round(v, 2)) for n, v in srt[:7]])
for n, want in {"spider web": 3.653, "garden spider": 3.595, "black and gold garden spider": 3.11, "barn spider": 2.27, "wolf spider": 2.046, "tick": 1.404}.items():
    check(f"unit 550 on photos of '{n}'", cls_mean[n], want, tol=0.03 * want)
check("median over the 100 other classes", float(np.median([cls_mean[NAMES[k]] for k in others])), 0.1178, tol=0.01)

In [ ]:
def sketch(kind, size=300):
    im = Image.new("RGB", (size, size), (255, 255, 255)); d = ImageDraw.Draw(im); c = size / 2
    if kind == "web":
        for i in range(12):
            a = 2 * math.pi * i / 12; d.line([(c, c), (c + 140 * math.cos(a), c + 140 * math.sin(a))], fill=0, width=3)
        for r in range(20, 141, 20):
            d.line([(c + r * math.cos(2 * math.pi * i / 12), c + r * math.sin(2 * math.pi * i / 12)) for i in range(13)], fill=0, width=2)
    elif kind == "spider":
        d.ellipse([c - 30, c - 22, c + 30, c + 38], fill=0); d.ellipse([c - 16, c - 52, c + 16, c - 20], fill=0)
        for s in (-1, 1):
            for ang in (-50, -15, 15, 50):
                a = math.radians(ang)
                d.line([(c + s * 28, c + 5 + ang * 0.3), (c + s * 85, c + 85 * math.sin(a) - 20), (c + s * 120, c + 120 * math.sin(a) + 30)], fill=0, width=5)
    elif kind == "sun":
        d.ellipse([c - 50, c - 50, c + 50, c + 50], outline=0, width=5)
        for i in range(12):
            a = 2 * math.pi * i / 12; d.line([(c + 65 * math.cos(a), c + 65 * math.sin(a)), (c + 120 * math.cos(a), c + 120 * math.sin(a))], fill=0, width=5)
    elif kind == "house":
        d.rectangle([c - 80, c - 10, c + 80, c + 110], outline=0, width=5); d.line([(c - 100, c - 10), (c, c - 110), (c + 100, c - 10)], fill=0, width=5)
    elif kind == "face":
        d.ellipse([c - 100, c - 110, c + 100, c + 110], outline=0, width=5)
        d.ellipse([c - 50, c - 40, c - 20, c - 10], fill=0); d.ellipse([c + 20, c - 40, c + 50, c - 10], fill=0)
        d.arc([c - 50, c, c + 50, c + 60], 20, 160, fill=0, width=5)
    elif kind == "grid":
        for i in range(-140, 141, 28):
            d.line([(c + i, c - 140), (c + i, c + 140)], fill=0, width=3); d.line([(c - 140, c + i), (c + 140, c + i)], fill=0, width=3)
    return im

kinds = ["spider", "web", "sun", "house", "face", "grid"]
pics = [sketch(k) for k in kinds]
_, a = classify(pics)
DRAW = {"web": 0.306, "spider": 2.246, "sun": 0.0, "house": 0.001, "face": 0.009, "grid": 0.043}
fig, axs = plt.subplots(1, 6, figsize=(11, 2.3))
for ax_, k, im, v in zip(axs, kinds, pics, a[:, 550]):
    ax_.imshow(im); ax_.axis("off"); ax_.set_title(f"{k}: {v:.2f}", fontsize=9)
    check(f"unit 550 on our drawing of a {k}", float(v), DRAW[k], tol=max(0.02, 0.03 * DRAW[k]))
fig.suptitle("unit 550 on line drawings", fontsize=10); plt.show()

A photo of a spider, a drawing of one, and the written word all reach the same neuron. Because CLIP read words with the
pictures, a word can also take over: paste a label on a fruit.

In [ ]:
def paste(im, text, frac=0.42):
    im = im.copy(); W_, H_ = im.size; w, h = int(W_ * frac), int(H_ * frac * 0.55)
    x0, y0 = (W_ - w) // 2, int(H_ * 0.52); d = ImageDraw.Draw(im)
    d.rectangle([x0, y0, x0 + w, y0 + h], fill=(246, 244, 236), outline=(180, 180, 175), width=1)
    if text:
        size = h; font = ImageFont.truetype(FONT, size)
        while size > 6 and d.textlength(text, font=font) > w * 0.86:
            size -= 1; font = ImageFont.truetype(FONT, size)
        d.text((x0 + (w - d.textlength(text, font=font)) / 2, y0 + (h - size) / 2 - size * 0.1), text, fill=(25, 25, 30), font=font)
    return im

OURS = {"Granny Smith | no label | font 0": [0.86, 0.0, -1], "Granny Smith | '' | font 0": [0.82, 0.0, -1], "Granny Smith | 'iPod' | font 0": [0.0, 1.0, -1], "Granny Smith | 'library' | font 0": [0.22, 0.0, 0.74], "Granny Smith | 'pizza' | font 0": [0.2, 0.0, 0.74], "Granny Smith | 'apple' | font 0": [0.96, 0.0, -1], "orange | no label | font 0": [0.74, 0.0, -1], "orange | '' | font 0": [0.76, 0.0, -1], "orange | 'iPod' | font 0": [0.0, 1.0, -1], "orange | 'library' | font 0": [0.34, 0.0, 0.58], "orange | 'pizza' | font 0": [0.14, 0.0, 0.84], "orange | 'apple' | font 0": [0.68, 0.0, -1]}
ip = NAMES.index("iPod")
shown = {}
for fruit in ("Granny Smith", "orange"):
    k = NAMES.index(fruit)
    ids = np.where(ALL_LABELS == k)[0].tolist(); fr = rows(ids)
    ims = [photo_pil(fr[n][0]) for n in ids]
    for cond in (None, "", "iPod", "library", "pizza", "apple"):
        test = ims if cond is None else [paste(im, cond) for im in ims]
        pr, _ = classify(test); pred = pr.argmax(1)
        tag = f"{fruit} | {'no label' if cond is None else repr(cond)} | font 0"
        acc, to_ip = float((pred == k).mean()), float((pred == ip).mean())
        extra = f", -> {cond} {100 * (pred == NAMES.index(cond)).mean():.0f}%" if cond in ("library", "pizza") else ""
        print(f"  {tag}: {fruit} {100 * acc:.0f}%, -> iPod {100 * to_ip:.0f}%{extra}")
        check(f"{fruit}, {'no label' if cond is None else repr(cond)}: right", acc, OURS[tag][0], tol=0.021)
        check(f"{fruit}, {'no label' if cond is None else repr(cond)}: called an iPod", to_ip, OURS[tag][1], tol=0.021)
        if fruit == "Granny Smith":
            shown[cond] = (test[0], NAMES[int(pred[0])])
fig, axs = plt.subplots(1, 6, figsize=(11, 2.5))
for ax_, (cond, (im, said)) in zip(axs, shown.items()):
    ax_.imshow(im); ax_.axis("off"); ax_.set_title(f"{'no label' if cond is None else repr(cond)} -> {said}", fontsize=8)
plt.show()

Every Granny Smith with the word "iPod" on it becomes an iPod. The same neurons that see the thing read the word.

## Your turn

1. **The other pole, drawn.** Draw 4e:55's *negative* pole: minimise its value instead of maximising it. What does it
   look like? *Hint: `drawing()` with the sign flipped, on `pre_relu("mixed4e", 55)`.*
2. **Pointing which way?** Using `stim` and `vals`, find the median value of V shapes for each pointing direction. Which
   directions does 4e:55 like? *Hint: `s[0][4]` is the direction.*
3. **A pure car unit.** Repeat section 4's vehicles-and-dogs count for 4d:417 at its top 1,000 photos. Does a "car" unit
   stay a car unit further down its list? *Hint: `series["4d:417"]`.*
4. **A word on an orange.** Paste "library" on the 50 oranges. Where do they go? *Hint: the loop in section 7.*

In [ ]:
# Your turn 1

In [ ]:
# Your turn 2

In [ ]:
# Your turn 3

In [ ]:
# Your turn 4

## Solutions

In [ ]:
# 1. The other pole, drawn
p55m = pre_relu("mixed4e", 55)
np.random.seed(0); torch.manual_seed(0)
spec, image = fft_image(1, seed=0); opt = torch.optim.Adam([spec], lr=0.05)
with Taps([p55m[0]]) as T:
    for step in range(512):
        opt.zero_grad(); net(prep(jiggle(image()))); (T[p55m[0]][0, p55m[1], 7, 7]).backward(); opt.step()
with torch.no_grad(), Taps([p55m[0]]) as T:
    x = image(); net(prep(x)); v = float(T[p55m[0]][0, p55m[1], 7, 7])
plt.figure(figsize=(3, 3)); plt.imshow(x[0].permute(1, 2, 0).cpu().numpy()); plt.axis("off"); plt.title(f"4e:55, minimised: {v:.0f}", fontsize=9); plt.show()
check("the minimised drawing goes lower than the typical photo's weakest place", bool(v < np.median(pmin)), True)
# The lecture's drawing of this pole is a round, flat, pug-like face.

In [ ]:
# 2. Pointing which way?
pts = {pd: float(np.median([v for s, v in zip(stim, vals) if s[0][0] == "V" and s[0][4] == pd])) for pd in range(0, 360, 30)}
print({k: round(v, 1) for k, v in pts.items()})
check("V pointing right (0 degrees): median", pts[0], 27.2, tol=0.3)
check("V pointing up (90 degrees): median", pts[90], -5.0, tol=0.3)
# Sideways and down, yes; pointing up, no.

In [ ]:
# 3. A pure car unit
idx = np.argsort(-series["4d:417"])[:1000]
print(f"4d:417, top 1,000: vehicles {frac(idx, VEHICLES):.2f}, dogs {frac(idx, DOGS):.2f}")
check("4d:417, top 1,000: vehicles", frac(idx, VEHICLES), {"4c:447": {"top40": [1.0, 0.0], "top200": [0.96, 0.0], "top1000": [0.777, 0.005]}, "4d:417": {"top40": [1.0, 0.0], "top200": [0.845, 0.085], "top1000": [0.387, 0.387]}, "4d:167": {"top40": [0.975, 0.025], "top200": [0.87, 0.03], "top1000": [0.492, 0.182]}, "4d:295": {"top40": [0.675, 0.3], "top200": [0.635, 0.36], "top1000": [0.4, 0.545]}, "4d:241": {"top40": [0.475, 0.525], "top200": [0.495, 0.49], "top1000": [0.32, 0.614]}, "4d:277": {"top40": [0.525, 0.0], "top200": [0.575, 0.015], "top1000": [0.389, 0.047]}}["4d:417"]["top1000"][0], tol=0.011)
check("4d:417, top 1,000: dogs", frac(idx, DOGS), {"4c:447": {"top40": [1.0, 0.0], "top200": [0.96, 0.0], "top1000": [0.777, 0.005]}, "4d:417": {"top40": [1.0, 0.0], "top200": [0.845, 0.085], "top1000": [0.387, 0.387]}, "4d:167": {"top40": [0.975, 0.025], "top200": [0.87, 0.03], "top1000": [0.492, 0.182]}, "4d:295": {"top40": [0.675, 0.3], "top200": [0.635, 0.36], "top1000": [0.4, 0.545]}, "4d:241": {"top40": [0.475, 0.525], "top200": [0.495, 0.49], "top1000": [0.32, 0.614]}, "4d:277": {"top40": [0.525, 0.0], "top200": [0.575, 0.015], "top1000": [0.389, 0.047]}}["4d:417"]["top1000"][1], tol=0.011)
# Further down its list, even the "car" unit fires on as many dogs as vehicles.

In [ ]:
# 4. A word on an orange
k = NAMES.index("orange"); ids = np.where(ALL_LABELS == k)[0].tolist(); fr = rows(ids)
pr, _ = classify([paste(photo_pil(fr[n][0]), "library") for n in ids]); pred = pr.argmax(1)
print(f"orange + 'library': orange {100 * (pred == k).mean():.0f}%, library {100 * (pred == NAMES.index('library')).mean():.0f}%")
check("oranges labelled 'library' that become a library", float((pred == NAMES.index("library")).mean()), {"Granny Smith | no label | font 0": [0.86, 0.0, -1], "Granny Smith | '' | font 0": [0.82, 0.0, -1], "Granny Smith | 'iPod' | font 0": [0.0, 1.0, -1], "Granny Smith | 'library' | font 0": [0.22, 0.0, 0.74], "Granny Smith | 'pizza' | font 0": [0.2, 0.0, 0.74], "Granny Smith | 'apple' | font 0": [0.96, 0.0, -1], "orange | no label | font 0": [0.74, 0.0, -1], "orange | '' | font 0": [0.76, 0.0, -1], "orange | 'iPod' | font 0": [0.0, 1.0, -1], "orange | 'library' | font 0": [0.34, 0.0, 0.58], "orange | 'pizza' | font 0": [0.14, 0.0, 0.84], "orange | 'apple' | font 0": [0.68, 0.0, -1]}["orange | 'library' | font 0"][2], tol=0.021)
print(f"whole notebook: {(time.time() - T0) / 60:.1f} minutes")